In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score, mean_absolute_percentage_error)
import xgboost as xgb

In [5]:
# ==========================================
# LANGKAH 1: Load & Persiapan Data
# ==========================================
df = pd.read_csv('modeling_ready.csv')

TARGET = 'y'
X = df.drop(columns=[TARGET])
y = df[TARGET]

kolom_kategorikal = X.select_dtypes(include=['object', 'category']).columns.tolist()
kolom_numerik     = X.select_dtypes(include=[np.number]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
# ==========================================
# LANGKAH 2: Preprocessor Tambahan
# ==========================================
preprocessor = ColumnTransformer(transformers=[
    ('num', MinMaxScaler(), kolom_numerik),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), kolom_kategorikal),
])

In [7]:
# ==========================================
# LANGKAH 3: Definisi Model + Hyperparameter
# ==========================================
kandidat_model = {
    'Baseline (LinearRegression)': {
        'model'  : LinearRegression(),
        'params' : {}
    },
    'ElasticNet': {
        'model'  : ElasticNet(max_iter=10000),
        'params' : {'model__alpha'   : [0.01, 0.1, 1, 10],
                    'model__l1_ratio': [0.2, 0.5, 0.8]}
    },
    'XGBoost': {
        'model'  : xgb.XGBRegressor(random_state=42, verbosity=0),
        'params' : {'model__n_estimators' : [100, 200],
                    'model__learning_rate': [0.05, 0.1],
                    'model__max_depth'    : [3, 5],
                    'model__subsample'    : [0.8, 1.0]}
    },
}

In [8]:
# ==========================================
# LANGKAH 4: Training + GridSearchCV + Evaluasi
# ==========================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'MAE' : 'neg_mean_absolute_error',
    'RMSE': 'neg_root_mean_squared_error',
    'R2'  : 'r2',
    'MAPE': 'neg_mean_absolute_percentage_error'
}

hasil        = {}
trained_models = {}  # ← menyimpan semua model yang sudah dilatih

for nama, config in kandidat_model.items():
    print(f"\n{'='*50}")
    print(f"  {nama}")
    print(f"{'='*50}")

    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model',        config['model'])
    ])

    if config['params']:
        grid_search = GridSearchCV(
            estimator  = pipe,
            param_grid = config['params'],
            cv         = kf,
            scoring    = 'neg_root_mean_squared_error',
            n_jobs     = -1,
            verbose    = 0
        )
        grid_search.fit(X_train, y_train)
        best_pipe = grid_search.best_estimator_
        print(f"  Best params : {grid_search.best_params_}")
    else:
        pipe.fit(X_train, y_train)
        best_pipe = pipe

    trained_models[nama] = best_pipe  # ← simpan tiap model setelah dilatih

    # ── Evaluasi Train-Test Split ──
    y_pred = best_pipe.predict(X_test)
    mae    = mean_absolute_error(y_test, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    r2     = r2_score(y_test, y_pred)
    mape   = mean_absolute_percentage_error(y_test, y_pred) * 100

    print(f"\n  [Train-Test Split]")
    print(f"  MAE  : {mae:.4f}")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  R2   : {r2:.4f}")
    print(f"  MAPE : {mape:.2f}%")

    # ── Evaluasi K-Fold ──
    cv_results = cross_validate(best_pipe, X, y, cv=kf, scoring=scoring)
    mae_cv  = -cv_results['test_MAE']
    rmse_cv = -cv_results['test_RMSE']
    r2_cv   =  cv_results['test_R2']
    mape_cv = -cv_results['test_MAPE'] * 100

    print(f"\n  [K-Fold Mean ± Std]")
    print(f"  MAE  : {mae_cv.mean():.4f} ± {mae_cv.std():.4f}")
    print(f"  RMSE : {rmse_cv.mean():.4f} ± {rmse_cv.std():.4f}")
    print(f"  R2   : {r2_cv.mean():.4f} ± {r2_cv.std():.4f}")
    print(f"  MAPE : {mape_cv.mean():.2f}% ± {mape_cv.std():.2f}%")

    hasil[nama] = {
        'MAE (Split)' : mae,
        'RMSE (Split)': rmse,
        'R2 (Split)'  : r2,
        'MAPE (Split)': round(mape, 2),
        'MAE (KFold)' : mae_cv.mean(),
        'RMSE (KFold)': rmse_cv.mean(),
        'R2 (KFold)'  : r2_cv.mean(),
        'MAPE (KFold)': round(mape_cv.mean(), 2),
    }


  Baseline (LinearRegression)

  [Train-Test Split]
  MAE  : 263.4912
  RMSE : 429.0581
  R2   : 0.1192
  MAPE : 197.32%

  [K-Fold Mean ± Std]
  MAE  : 264.4264 ± 2.0228
  RMSE : 427.5767 ± 6.8868
  R2   : 0.1125 ± 0.0053
  MAPE : 203.09% ± 13.10%

  ElasticNet
  Best params : {'model__alpha': 0.01, 'model__l1_ratio': 0.8}

  [Train-Test Split]
  MAE  : 263.2975
  RMSE : 430.1024
  R2   : 0.1149
  MAPE : 198.67%

  [K-Fold Mean ± Std]
  MAE  : 264.3432 ± 2.2204
  RMSE : 428.1501 ± 7.2832
  R2   : 0.1102 ± 0.0039
  MAPE : 202.63% ± 9.70%

  XGBoost
  Best params : {'model__learning_rate': 0.1, 'model__max_depth': 5, 'model__n_estimators': 200, 'model__subsample': 0.8}

  [Train-Test Split]
  MAE  : 226.2134
  RMSE : 369.7664
  R2   : 0.3458
  MAPE : 152.94%

  [K-Fold Mean ± Std]
  MAE  : 227.3926 ± 1.1204
  RMSE : 371.7739 ± 4.2523
  R2   : 0.3288 ± 0.0163
  MAPE : 161.14% ± 13.25%


In [9]:
# ==========================================
# LANGKAH 5: Tabel Perbandingan Utama
# ==========================================
print(f"\n\n{'='*75}")
print("  PERBANDINGAN SEMUA MODEL")
print(f"{'='*75}")

df_hasil = pd.DataFrame(hasil).T.round(4)
print(df_hasil.to_string())

best_model = df_hasil['R2 (KFold)'].idxmax()
best_r2    = df_hasil['R2 (KFold)'].max()
print(f"\nModel terbaik : {best_model} (R2 K-Fold = {best_r2:.4f})")



  PERBANDINGAN SEMUA MODEL
                             MAE (Split)  RMSE (Split)  R2 (Split)  MAPE (Split)  MAE (KFold)  RMSE (KFold)  R2 (KFold)  MAPE (KFold)
Baseline (LinearRegression)     263.4912      429.0581      0.1192        197.32     264.4264      427.5767      0.1125        203.09
ElasticNet                      263.2975      430.1024      0.1149        198.67     264.3432      428.1501      0.1102        202.63
XGBoost                         226.2134      369.7664      0.3458        152.94     227.3926      371.7739      0.3288        161.14

Model terbaik : XGBoost (R2 K-Fold = 0.3288)


In [10]:
# ==========================================
# LANGKAH 6: Verifikasi trained_models
# (Pastikan tiap model tersimpan dengan benar)
# ==========================================
print(f"\n{'='*50}")
print("  Verifikasi Model Tersimpan")
print(f"{'='*50}")
for nama, pipe in trained_models.items():
    tipe = type(pipe.named_steps['model']).__name__
    print(f"  {nama:<35} → {tipe}")


  Verifikasi Model Tersimpan
  Baseline (LinearRegression)         → LinearRegression
  ElasticNet                          → ElasticNet
  XGBoost                             → XGBRegressor


In [11]:
# ==========================================
# LANGKAH 7: STRESS TEST dengan Data Noisy
# ==========================================
print(f"\n\n{'='*75}")
print("  STRESS TEST: Perbandingan Data Asli vs Data Noisy")
print(f"{'='*75}")

# Muat data noisy dari temanmu
# ↓ Ganti nama file sesuai file dari temanmu
df_noisy = pd.read_csv('dataset_noise_stress_city.csv')

X_noisy = df_noisy.drop(columns=[TARGET])
y_noisy = df_noisy[TARGET]

hasil_stress = {}

for nama, pipe in trained_models.items():
    print(f"\n{'='*50}")
    print(f"  {nama}")
    print(f"{'='*50}")

    # Prediksi data asli & data noisy
    y_pred_asli  = pipe.predict(X_test)
    y_pred_noisy = pipe.predict(X_noisy)

    # Metrik data asli
    mae_a  = mean_absolute_error(y_test, y_pred_asli)
    rmse_a = np.sqrt(mean_squared_error(y_test, y_pred_asli))
    r2_a   = r2_score(y_test, y_pred_asli)
    mape_a = mean_absolute_percentage_error(y_test, y_pred_asli) * 100

    # Metrik data noisy
    mae_n  = mean_absolute_error(y_noisy, y_pred_noisy)
    rmse_n = np.sqrt(mean_squared_error(y_noisy, y_pred_noisy))
    r2_n   = r2_score(y_noisy, y_pred_noisy)
    mape_n = mean_absolute_percentage_error(y_noisy, y_pred_noisy) * 100

    print(f"\n  {'Metrik':<8} {'Data Asli':>12} {'Data Noisy':>12} {'Selisih':>12} {'Status':>12}")
    print(f"  {'-'*60}")

    for metrik, v_a, v_n in [("MAE", mae_a, mae_n), ("RMSE", rmse_a, rmse_n),
                               ("R2", r2_a, r2_n),   ("MAPE", mape_a, mape_n)]:
        selisih = v_n - v_a
        if metrik == "R2":
            status = "Stabil" if abs(selisih) < 0.05 else "Sensitif"
        else:
            status = "Stabil" if abs(selisih) / (v_a + 1e-9) < 0.05 else "Sensitif"
        unit = "%" if metrik == "MAPE" else ""
        print(f"  {metrik:<8} {v_a:>11.4f}{unit} {v_n:>11.4f}{unit} {selisih:>+11.4f} {status:>12}")

    hasil_stress[nama] = {
        'MAE (Asli)'  : round(mae_a, 4),  'MAE (Noisy)' : round(mae_n, 4),
        'RMSE (Asli)' : round(rmse_a, 4), 'RMSE (Noisy)': round(rmse_n, 4),
        'R2 (Asli)'   : round(r2_a, 4),   'R2 (Noisy)'  : round(r2_n, 4),
        'MAPE (Asli)' : round(mape_a, 2), 'MAPE (Noisy)': round(mape_n, 2),
    }

# ── Tabel Ringkasan Stress Test ──
print(f"\n\n{'='*75}")
print("  RINGKASAN STRESS TEST")
print(f"{'='*75}")

df_stress = pd.DataFrame(hasil_stress).T
print(df_stress.to_string())

# Model paling tahan noise berdasarkan selisih R2 terkecil
selisih_r2 = {nama: abs(v['R2 (Noisy)'] - v['R2 (Asli)']) for nama, v in hasil_stress.items()}
model_robust = min(selisih_r2, key=selisih_r2.get)
print(f"\nModel paling tahan noise : {model_robust} (ΔR2 = {selisih_r2[model_robust]:.4f})")



  STRESS TEST: Perbandingan Data Asli vs Data Noisy

  Baseline (LinearRegression)

  Metrik      Data Asli   Data Noisy      Selisih       Status
  ------------------------------------------------------------
  MAE         263.4912    272.3031     +8.8119       Stabil
  RMSE        429.0581    434.4298     +5.3717       Stabil
  R2            0.1192      0.0843     -0.0349       Stabil
  MAPE        197.3231%    204.2749%     +6.9518       Stabil

  ElasticNet

  Metrik      Data Asli   Data Noisy      Selisih       Status
  ------------------------------------------------------------
  MAE         263.2975    266.4105     +3.1130       Stabil
  RMSE        430.1024    429.0228     -1.0795       Stabil
  R2            0.1149      0.1069     -0.0080       Stabil
  MAPE        198.6665%    205.8349%     +7.1685       Stabil

  XGBoost

  Metrik      Data Asli   Data Noisy      Selisih       Status
  ------------------------------------------------------------
  MAE         226.2134   